In [ ]:
import pandas as pd
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import median_abs_deviation, pearsonr, uniform, randint # Import distributions for Randomized Search
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.model_selection import RandomizedSearchCV # Import RandomizedSearchCV

# Feature Ranking Methods (Keep as is)
def chi2_rank(X, y):
    print("Computing Chi-squared statistics...")
    X_non_negative = X.copy()
    min_val = X_non_negative.min().min()
    if min_val < 0:
        X_non_negative = X_non_negative - min_val
    X_non_negative = X_non_negative.replace([np.inf, -np.inf], np.nan).fillna(0)
    variances = X_non_negative.var()
    cols_with_variance = variances[variances > 1e-9].index
    X_filtered = X_non_negative[cols_with_variance]
    if X_filtered.empty:
        print("Warning: No features with variance > 0 for Chi-squared.")
        return pd.Series(0.0, index=X.columns).fillna(0)
    try:
        chi2_scores, _ = chi2(X_filtered, y)
        scores = pd.Series(0.0, index=X.columns)
        scores[cols_with_variance] = chi2_scores
        return scores.fillna(0)
    except Exception as e:
        print(f"Error during Chi-squared calculation: {e}")
        return pd.Series(0.0, index=X.columns).fillna(0)

def mad_rank(X):
    print("Computing Median Absolute Deviation...")
    X_numeric = X.select_dtypes(include=np.number).replace([np.inf, -np.inf], np.nan).fillna(0)
    if X_numeric.empty:
        print("Warning: No numerical features for MAD.")
        return pd.Series(dtype=float)
    mad_scores = pd.Series(median_abs_deviation(X_numeric, axis=0), index=X_numeric.columns).fillna(0)
    return mad_scores

def pcc_rank(X, y):
    print("Computing Pearson Correlation Coefficients...")
    X_numeric = X.select_dtypes(include=np.number).replace([np.inf, -np.inf], np.nan).fillna(0)
    if X_numeric.empty:
        print("Warning: No numerical features for PCC.")
        return pd.Series(dtype=float)
    pcc_scores = {}
    for col in X_numeric.columns:
        if X_numeric[col].nunique() > 1 and X_numeric[col].var() > 1e-9:
             try:
                 corr, _ = pearsonr(X_numeric[col], y)
                 pcc_scores[col] = abs(corr)
             except Exception as e:
                 pcc_scores[col] = 0
        else:
             pcc_scores[col] = 0
    return pd.Series(pcc_scores).fillna(0)

def mi_rank(X, y):
    print("Computing Mutual Information...")
    X_cleaned = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    try:
        mi_scores = mutual_info_classif(X_cleaned, y, discrete_features='auto', random_state=42)
        return pd.Series(mi_scores, index=X_cleaned.columns).fillna(0)
    except Exception as e:
        print(f"Error during Mutual Information calculation: {e}")
        return pd.Series(0.0, index=X.columns).fillna(0)

def lgbm_rank(X, y):
    print("Computing LightGBM feature importance...")
    X_cleaned = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    if X_cleaned.empty:
         print("Warning: Empty DataFrame for LGBM ranking.")
         return pd.Series(dtype=float)
    try:
        model = LGBMClassifier(n_estimators=100, verbose=-1, random_state=42, n_jobs=-1)
        model.fit(X_cleaned, y)
        return pd.Series(model.feature_importances_, index=X_cleaned.columns).fillna(0)
    except Exception as e:
        print(f"Error during LightGBM feature importance calculation: {e}")
        return pd.Series(0.0, index=X.columns).fillna(0)

def normalize(series):
    if series.empty:
        return pd.Series(dtype=float)
    min_val = series.min()
    max_val = series.max()
    range_val = max_val - min_val
    if range_val == 0:
        return pd.Series(0.0, index=series.index)
    return (series - min_val) / (range_val + 1e-6)

# Fuzzy TOPSIS (Keep as is)
def fuzzy_topsis(df, weights=None):
    print("Applying Fuzzy TOPSIS for feature ranking...")
    if df.empty:
        print("Warning: Empty DataFrame received by fuzzy_topsis.")
        return pd.Series(dtype=float)
    df_numeric = df.select_dtypes(include=np.number).fillna(0)
    if df_numeric.empty:
        print("Warning: No numeric columns in DataFrame received by fuzzy_topsis.")
        return pd.Series(dtype=float)
    norm = np.sqrt((df_numeric**2).sum())
    norm = norm.replace(0, 1e-6)
    norm_df = df_numeric / norm
    weights = weights or [1/norm_df.shape[1]] * norm_df.shape[1]
    if len(weights) != norm_df.shape[1]:
         print(f"Warning: Number of weights ({len(weights)}) does not match number of features ({norm_df.shape[1]}). Using equal weights.")
         weights = [1/norm_df.shape[1]] * norm_df.shape[1]
    weighted_df = norm_df * weights
    ideal_best = weighted_df.max()
    ideal_worst = weighted_df.min()
    d_best = np.sqrt(((weighted_df - ideal_best)**2).sum(axis=1))
    d_worst = np.sqrt(((weighted_df - ideal_worst)**2).sum(axis=1))
    closeness = d_worst / (d_best + d_worst + 1e-6)
    result = closeness.sort_values(ascending=False)
    if not result.empty:
         print(f"Top 10 features from TOPSIS: {', '.join(result.head(10).index.tolist())}")
    else:
         print("TOPSIS result is empty.")
    return result

# Improved Grey Wolf Optimizer (impGWO) (Keep logic, maybe tune internal model slightly)
def impgwo_optimize(X_train, y_train, X_test, y_test, feat_list, iterations=15, wolves=20):
    print("\n" + "="*50)
    print(f"Starting Improved Grey Wolf Optimizer with {wolves} wolves and {iterations} iterations")
    print(f"Initial feature set size (input to GWO): {len(feat_list)}")
    print("="*50)

    start_time = time.time()
    dim = len(feat_list)

    if dim == 0:
        print("No features provided to GWO.")
        return []

    wolves_pos = np.random.rand(wolves, dim)
    scores = np.full(wolves, -1.0)

    best_fitness_history = []
    iteration_history = []

    full_X_train = X_train
    full_X_test = X_test

    print("\nEvaluating initial wolf positions...")
    for i in range(wolves):
        binary = wolves_pos[i] > 0.5
        selected = [feat_list[j] for j in range(dim) if binary[j]]
        if not selected:
            scores[i] = -1.0
            continue
        try:
            # Slightly simpler internal model for speed
            model = XGBClassifier(n_estimators=30, max_depth=3, learning_rate=0.1,
                                  subsample=0.8, colsample_bytree=0.8,
                                  use_label_encoder=False, eval_metric='logloss', verbosity=0,
                                  random_state=42, n_jobs=1) # Use n_jobs=1 inside GWO
            model.fit(full_X_train[selected], y_train)
            acc = accuracy_score(y_test, model.predict(full_X_test[selected]))
            scores[i] = acc
        except Exception as e:
             # print(f"Error evaluating wolf {i+1} initial pos: {e}") # Too verbose
             scores[i] = -1.0

    valid_scores_indices = np.where(scores != -1.0)[0]
    if len(valid_scores_indices) == 0:
        print("Warning: All initial wolves resulted in invalid feature sets or errors. Returning all input features.")
        return feat_list

    sorted_indices = valid_scores_indices[np.argsort(scores[valid_scores_indices])[::-1]]

    alpha_idx = sorted_indices[0]
    alpha_pos = wolves_pos[alpha_idx].copy()
    alpha_score = scores[alpha_idx]
    alpha_features = sum(alpha_pos > 0.5)

    beta_idx = sorted_indices[1] if len(sorted_indices) > 1 else alpha_idx
    beta_pos = wolves_pos[beta_idx].copy()
    beta_score = scores[beta_idx]
    beta_features = sum(beta_pos > 0.5)

    delta_idx = sorted_indices[2] if len(sorted_indices) > 2 else beta_idx
    delta_pos = wolves_pos[delta_idx].copy()
    delta_score = scores[delta_idx]
    delta_features = sum(delta_pos > 0.5)

    print("\nInitial leaders:")
    print(f"Alpha wolf (Best Fitness): {alpha_features} features, score: {alpha_score:.4f}")
    print(f"Beta wolf:  {beta_features} features, score: {beta_score:.4f}")
    print(f"Delta wolf: {delta_features} features, score: {delta_score:.4f}")

    best_fitness_history.append(alpha_score)
    iteration_history.append(0)

    T = iterations

    for t in range(iterations):
        k = 2 * (1 - ((t+1)**1.5 / T**1.5))
        print(f"\nIteration {t+1}/{iterations} (k = {k:.4f}): Current Best Fitness: {alpha_score:.4f} ({alpha_features} features)")

        current_alpha_pos = alpha_pos.copy()
        current_beta_pos = beta_pos.copy()
        current_delta_pos = delta_pos.copy()

        for i in range(wolves):
            A1 = 2 * k * random.random() - k
            C1 = 2 * random.random()
            A2 = 2 * k * random.random() - k
            C2 = 2 * random.random()
            A3 = 2 * k * random.random() - k
            C3 = 2 * random.random()

            D_alpha = abs(C1 * current_alpha_pos - wolves_pos[i])
            D_beta = abs(C2 * current_beta_pos - wolves_pos[i])
            D_delta = abs(C3 * current_delta_pos - wolves_pos[i])

            X1 = current_alpha_pos - A1 * D_alpha
            X2 = current_beta_pos - A2 * D_beta
            X3 = current_delta_pos - A3 * D_delta

            wolves_pos[i] = (X1 + X2 + X3) / 3.0
            wolves_pos[i] = np.clip(wolves_pos[i], 0, 1)

            binary = wolves_pos[i] > 0.5
            selected = [feat_list[j] for j in range(dim) if binary[j]]
            if not selected:
                 scores[i] = -1.0
                 continue

            try:
                model = XGBClassifier(n_estimators=30, max_depth=3, learning_rate=0.1,
                                      subsample=0.8, colsample_bytree=0.8,
                                      use_label_encoder=False, eval_metric='logloss', verbosity=0,
                                      random_state=42, n_jobs=1)
                model.fit(full_X_train[selected], y_train)
                acc = accuracy_score(y_test, model.predict(full_X_test[selected]))
                scores[i] = acc

                valid_scores_indices_iter = np.where(scores != -1.0)[0]
                if len(valid_scores_indices_iter) > 0:
                    sorted_indices_iter = valid_scores_indices_iter[np.argsort(scores[valid_scores_indices_iter])[::-1]]
                    new_alpha_idx = sorted_indices_iter[0]
                    new_beta_idx = sorted_indices_iter[1] if len(sorted_indices_iter) > 1 else new_alpha_idx
                    new_delta_idx = sorted_indices_iter[2] if len(sorted_indices_iter) > 2 else (new_beta_idx if len(sorted_indices_iter) > 1 else new_alpha_idx)

                    if scores[new_alpha_idx] > alpha_score:
                        delta_pos = beta_pos.copy()
                        delta_score = beta_score
                        delta_features = beta_features
                        beta_pos = alpha_pos.copy()
                        beta_score = alpha_score
                        beta_features = alpha_features
                        alpha_pos = wolves_pos[new_alpha_idx].copy()
                        alpha_score = scores[new_alpha_idx]
                        alpha_features = sum(alpha_pos > 0.5)
                        print(f"New BEST FITNESS found! Wolf {new_alpha_idx+1} in iter {t+1}: {alpha_features} features, score: {alpha_score:.4f}")
                    elif scores[i] > beta_score and i != alpha_idx:
                         if i != alpha_idx and i != delta_idx:
                              delta_pos = beta_pos.copy()
                               delta_score = beta_score
                              delta_features = beta_features
                              beta_pos = wolves_pos[i].copy()
                              beta_score = scores[i]
                              beta_features = sum(binary)
                              print(f"New beta found!  Wolf {i+1} in iter {t+1}: {beta_features} features, score: {beta_score:.4f}")
                    elif scores[i] > delta_score and i != alpha_idx and i != beta_idx:
                         delta_pos = wolves_pos[i].copy()
                         delta_score = scores[i]
                         delta_features = sum(binary)
                         print(f"New delta found! Wolf {i+1} in iter {t+1}: {delta_features} features, score: {delta_score:.4f}")

            except Exception as e:
                # print(f"Error evaluating wolf {i+1} after update in iter {t+1}: {e}") # Too verbose
                scores[i] = -1.0

        valid_scores_indices_end_iter = np.where(scores != -1.0)[0]
        if len(valid_scores_indices_end_iter) > 0:
             sorted_indices_end_iter = valid_scores_indices_end_iter[np.argsort(scores[valid_scores_indices_end_iter])[::-1]]
             alpha_score = scores[sorted_indices_end_iter[0]]
             alpha_pos = wolves_pos[sorted_indices_end_iter[0]].copy()
             alpha_features = sum(alpha_pos > 0.5)
             if len(sorted_indices_end_iter) > 1:
                  beta_pos = wolves_pos[sorted_indices_end_iter[1]].copy()
                  beta_score = scores[sorted_indices_end_iter[1]]
                  beta_features = sum(beta_pos > 0.5)
             if len(sorted_indices_end_iter) > 2:
                  delta_pos = wolves_pos[sorted_indices_end_iter[2]].copy()
                  delta_score = scores[sorted_indices_end_iter[2]]
                  delta_features = sum(delta_pos > 0.5)

        best_fitness_history.append(alpha_score)
        iteration_history.append(t+1)

        print(f"End of iteration {t+1} summary:")
        print(f"Alpha wolf (BEST FITNESS): {alpha_features} features, score: {alpha_score:.4f}")
        print(f"Beta wolf:  {beta_features} features, score: {beta_score:.4f}")
        print(f"Delta wolf: {delta_features} features, score: {delta_score:.4f}")

    elapsed_time = time.time() - start_time
    print("\n" + "="*50)
    print(f"Improved GWO completed in {elapsed_time:.2f} seconds")
    print(f"Best fitness evolution: {', '.join([f'{score:.4f}' for score in best_fitness_history])}")

    try:
        plt.figure(figsize=(10, 6))
        plt.plot(iteration_history, best_fitness_history, marker='o', linestyle='-', color='blue')
        plt.title('Best Fitness Evolution During impGWO Optimization')
        plt.xlabel('Iteration')
        plt.ylabel('Best Fitness (Accuracy)')
        plt.grid(True)
        plt.savefig('impGWO_fitness_history.png')
        plt.close()
        print("Best fitness evolution plot saved as 'impGWO_fitness_history.png'")
    except Exception as e:
        print(f"Could not save plot: {e}")

    if alpha_pos is None or alpha_score == -1.0:
        print("No valid alpha wolf found after GWO. Returning all input features to GWO.")
        return feat_list

    best_binary = alpha_pos > 0.5
    selected_features = [feat_list[i] for i in range(dim) if binary[i]] # Use alpha_pos's binary selection
    selected_features_check = [feat_list[i] for i in range(dim) if alpha_pos[i] > 0.5] # Double check with alpha_pos

    if not selected_features_check:
        print("Alpha wolf selected no features. Returning all input features to GWO.")
        return feat_list

    selected_features = selected_features_check # Use the correctly filtered list

    print(f"Final selection from GWO: {len(selected_features)} features with accuracy (best fitness) {alpha_score:.4f}")
    if len(selected_features) <= 10:
        print(f"Selected features: {', '.join(selected_features)}")
    else:
        print(f"Selected features (showing 10 of {len(selected_features)}): {', '.join(selected_features[:10])}...")
    print("="*50)

    return selected_features


# Evaluation (Keep as is)
def evaluate(y_true, y_pred):
    print("\n" + "="*50)
    print("Model Evaluation Results:")
    print("="*50)
    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    if cm.size == 0:
        specificity = 0.0
    else:
        FP = cm.sum(axis=0) - np.diag(cm)
        FN = cm.sum(axis=1) - np.diag(cm)
        TP = np.diag(cm)
        TN = cm.sum() - (FP + FN + TP)
        specificity_per_class = np.divide(TN, (TN + FP), out=np.zeros_like(TN, dtype=float), where=(TN + FP) != 0)
        specificity = np.mean(specificity_per_class)
    print(f"Specificity: {specificity:.4f}")
    print(f"Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision:   {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall:      {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 Score:    {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print("="*50)

# Preprocessing (Corrected Target Encoding)
def preprocess(train_csv, test_csv):
    print("\n" + "="*50)
    print(f"Loading data from {train_csv} and {test_csv}")
    start_time = time.time()
    try:
        df_train = pd.read_csv(train_csv)
        df_test = pd.read_csv(test_csv)
    except FileNotFoundError as e:
        print(f"Error loading data: {e}")
        return None, None, None, None, None
    print(f"Training set: {df_train.shape[0]} samples, {df_train.shape[1]} features")
    print(f"Testing set:  {df_test.shape[0]} samples, {df_test.shape[1]} features")

    categorical_cols = ['protocol_type', 'service', 'flag']
    label_col = 'labels'

    required_cols = categorical_cols + [label_col]
    for col in required_cols:
        if col not in df_train.columns:
            print(f"Error: Missing required training column '{col}'.")
            return None, None, None, None, None
        if col not in df_test.columns:
             print(f"Error: Missing required testing column '{col}'.")
             return None, None, None, None, None

    train_cols = set(df_train.columns)
    test_cols = set(df_test.columns)
    unique_to_train = list(train_cols - test_cols - {label_col})
    unique_to_test = list(test_cols - train_cols - {label_col})
    if unique_to_train:
        print(f"Warning: Dropping columns unique to train set: {unique_to_train}")
        df_train = df_train.drop(columns=unique_to_train)
    if unique_to_test:
        print(f"Warning: Dropping columns unique to test set: {unique_to_test}")
        df_test = df_test.drop(columns=unique_to_test)
    common_features = [col for col in df_train.columns if col != label_col]
    df_test = df_test[common_features + [label_col]].copy() # Use copy

    print("\nEncoding categorical features...")
    for col in categorical_cols:
        if col in df_train.columns:
            combined_categories = pd.concat([df_train[col], df_test[col]]).astype('category').cat.categories
            df_train[col] = pd.Categorical(df_train[col], categories=combined_categories).codes
            df_test[col] = pd.Categorical(df_test[col], categories=combined_categories).codes
            if (df_train[col] == -1).any() or (df_test[col] == -1).any():
                 print(f"Warning: Unseen categories resulted in -1 codes for '{col}'. Filling -1 with 0.")
                 df_train[col] = df_train[col].replace(-1, 0)
                 df_test[col] = df_test[col].replace(-1, 0)

    print("Encoding target variable...")
    # Step 1: Initial Label Encoding based on ALL unique labels across train and test
    target_le_initial = LabelEncoder()
    combined_original_labels = pd.concat([df_train[label_col], df_test[label_col]]).unique()
    target_le_initial.fit(combined_original_labels)
    df_train[label_col] = target_le_initial.transform(df_train[label_col])
    df_test[label_col] = target_le_initial.transform(df_test[label_col])

    # Step 2: Filter test set to only include classes seen in the *transformed* training data
    train_encoded_classes_present = np.unique(df_train[label_col])
    df_test = df_test[df_test[label_col].isin(train_encoded_classes_present)].copy()

    X_train = df_train.drop(label_col, axis=1)
    y_train = df_train[label_col].copy() # Use copy
    X_test = df_test.drop(label_col, axis=1)
    y_test = df_test[label_col].copy() # Use copy

    # Step 3: Re-encode y_train and y_test to be 0-indexed and contiguous (FIX for XGBoost ValueError)
    print("Finalizing target variable encoding (0-indexed, contiguous)...")
    y_train_final_encoded, train_classes_0_indexed = pd.factorize(y_train)
    y_train_final_encoded = pd.Series(y_train_final_encoded, index=y_train.index, name=label_col)

    original_int_to_zero_indexed_map = {original_int: new_int for new_int, original_int in enumerate(train_classes_0_indexed)}

    y_test_final_encoded = y_test.map(original_int_to_zero_indexed_map)
    nan_indices = y_test_final_encoded[y_test_final_encoded.isna()].index
    if not nan_indices.empty:
         print(f"Warning: Dropping {len(nan_indices)} test samples with labels not in training set after mapping.")
         X_test = X_test.drop(index=nan_indices)
         y_test_final_encoded = y_test_final_encoded.drop(index=nan_indices)
    y_test_final_encoded = pd.Series(y_test_final_encoded, index=X_test.index, name=label_col)

    print(f"Number of final classes in training data: {len(train_classes_0_indexed)}")
    print(f"Number of final classes in testing data: {y_test_final_encoded.nunique()}")

    numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    print("Scaling numerical features...")
    scaler = MinMaxScaler()
    if numerical_cols:
        X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
        X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])
    else:
        print("No numerical columns found for scaling.")

    print("Checking class distribution...")
    class_counts = Counter(y_train_final_encoded)
    print(f"Class distribution before balancing: {dict(class_counts)}")
    smote_k_neighbors = 5
    min_class_size = min(class_counts.values()) if len(class_counts) > 0 else 0

    if len(train_classes_0_indexed) > 1 and min_class_size > smote_k_neighbors:
        print(f"Applying SMOTE to balance classes (k_neighbors={smote_k_neighbors})...")
        try:
            smote = SMOTE(k_neighbors=smote_k_neighbors, random_state=42, sampling_strategy='auto')
            X_train_res, y_train_res = smote.fit_resample(X_train, y_train_final_encoded)
            X_train = pd.DataFrame(X_train_res, columns=X_train.columns)
            final_y_train = pd.Series(y_train_res, name=label_col)
            print(f"Class distribution after balancing: {dict(Counter(final_y_train))}")
        except Exception as e:
            print(f"SMOTE failed: {e}. Proceeding without SMOTE.")
            print("SMOTE requires min class size > k_neighbors. Check your data distribution.")
            final_y_train = y_train_final_encoded
    else:
        print(f"Skipping SMOTE. Unique classes: {len(train_classes_0_indexed)}, Min class size: {min_class_size}")
        final_y_train = y_train_final_encoded

    unique_classes_after_smote = np.unique(final_y_train)
    if len(unique_classes_after_smote) < 2:
         print("Error: Training data must contain at least two classes after preprocessing.")
         return None, None, None, None, None

    original_label_names = target_le_initial.inverse_transform(train_classes_0_indexed)

    elapsed_time = time.time() - start_time
    print(f"Preprocessing completed in {elapsed_time:.2f} seconds")
    print("="*50)
    return X_train, final_y_train, X_test, y_test_final_encoded, original_label_names


# Main Execution (Using RandomizedSearchCV)
def main(train_file, test_file, top_k=30):
    print("\n" + "="*50)
    print(f"Starting feature selection and model training process with initial top_k={top_k}")
    print("="*50)

    overall_start_time = time.time()
    X_train, y_train, X_test, y_test, class_names = preprocess(train_file, test_file)

    if X_train is None:
        print("Preprocessing failed. Exiting.")
        return

    print("\nApplying multiple feature ranking methods...")
    X_train_numeric = X_train.select_dtypes(include=np.number)

    chi2_scores = normalize(chi2_rank(X_train_numeric, y_train))
    mad_scores = normalize(mad_rank(X_train))
    pcc_scores = normalize(pcc_rank(X_train, y_train))
    mi_scores = normalize(mi_rank(X_train, y_train))
    lgbm_scores = normalize(lgbm_rank(X_train, y_train))

    print("\nCombining ranking scores for TOPSIS...")
    all_score_series = [chi2_scores, mad_scores, pcc_scores, mi_scores, lgbm_scores]
    all_indices = [scores.index for scores in all_score_series if not scores.empty]

    initial_gwo_features = []
    scores_df = pd.DataFrame()

    if not all_indices:
        print("Error: No valid feature score indices returned from any ranker.")
        print("Using all features for GWO as fallback.")
        initial_gwo_features = X_train.columns.tolist()
    else:
        all_ranked_features_union = set.union(*map(set, all_indices))
        final_feature_pool_for_topsis = list(all_ranked_features_union.intersection(X_train.columns))

        if not final_feature_pool_for_topsis:
            print("Error: Union of ranked features has no intersection with training columns.")
            print("Using all features for GWO as fallback.")
            initial_gwo_features = X_train.columns.tolist()
        else:
            scores_data = {}
            ranker_names = ['Chi2', 'MAD', 'PCC', 'MI', 'LGBM']
            for i, ranker_name in enumerate(ranker_names):
                 if not all_score_series[i].empty:
                     scores_data[ranker_name] = all_score_series[i].reindex(final_feature_pool_for_topsis).fillna(0)
                 else:
                     scores_data[ranker_name] = pd.Series(0.0, index=final_feature_pool_for_topsis)
            scores_df = pd.DataFrame(scores_data)

            if scores_df.empty:
                 print("Critical Error: scores_df is empty after union logic. Cannot proceed with TOPSIS.")
                 print("Using all features for GWO as fallback.")
                 initial_gwo_features = X_train.columns.tolist()
            else:
                topsis_rank = fuzzy_topsis(scores_df)
                if topsis_rank.empty:
                     print("Error: TOPSIS returned an empty ranking.")
                     print("Using all features for GWO as fallback.")
                     initial_gwo_features = X_train.columns.tolist()
                else:
                    actual_top_k = min(top_k, topsis_rank.shape[0])
                    initial_gwo_features = topsis_rank.head(actual_top_k).index.tolist()
                    if not initial_gwo_features:
                        print(f"Error: TOPSIS selected no features from the {scores_df.shape[0]} considered features.")
                        print("Using all features for GWO as fallback.")
                        initial_gwo_features = X_train.columns.tolist()
                    else:
                         print(f"TOPSIS successfully selected {len(initial_gwo_features)} features for GWO input.")

    selected_features = impgwo_optimize(X_train, y_train, X_test, y_test, initial_gwo_features, iterations=15, wolves=20)

    if not selected_features:
         print("GWO returned no features. Cannot train model.")
         print("Final Fallback: Using all original features for training.")
         selected_features = X_train.columns.tolist()
         if not selected_features:
             print("No features available in X_train. Exiting.")
             return

    print("\n" + "="*50)
    print(f"Training final model with {len(selected_features)} selected features using Hyperparameter Tuning (RandomizedSearchCV)")
    print("="*50)

    # Define the parameter distribution for RandomizedSearchCV
    # Using distributions allows sampling from a range (e.g., randint, uniform)
    param_distributions = {
        'n_estimators': randint(100, 500), # Random integer between 100 and 500
        'max_depth': randint(4, 12),      # Random integer between 4 and 12
        'learning_rate': uniform(0.01, 0.1), # Random float between 0.01 and 0.1 (0.01 + 0.1)
        'subsample': uniform(0.7, 0.3),   # Random float between 0.7 and 1.0 (0.7 + 0.3)
        'colsample_bytree': uniform(0.7, 0.3) # Random float between 0.7 and 1.0
    }

    # Initialize the base model
    # use_label_encoder=False requires 0-indexed contiguous integer labels (handled in preprocess)
    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', verbosity=0, random_state=42, n_jobs=-1)

    # Initialize RandomizedSearchCV
    # n_iter controls the number of parameter combinations sampled (tune for speed/accuracy)
    # cv_value adjusted based on min class size in preprocess
    cv_value = max(2, min(Counter(y_train).values()) if len(Counter(y_train)) > 0 else 2) # Ensure at least cv=2 if possible
    if cv_value < 2 and len(np.unique(y_train)) > 1: cv_value = 2 # Minimum 2 folds for meaningful CV

    n_iter_search = 100 # Number of parameter settings that are sampled (adjust this!)

    print(f"Using cv={cv_value} for RandomizedSearchCV.")
    print(f"Sampling {n_iter_search} parameter combinations.")

    random_search = RandomizedSearchCV(estimator=xgb, param_distributions=param_distributions,
                                       n_iter=n_iter_search, scoring='accuracy', cv=cv_value,
                                       n_jobs=-1, verbose=1, random_state=42)

    print(f"Starting Randomized Search on {len(selected_features)} selected features...")
    tuning_start_time = time.time()

    clf = None
    best_params = "N/A"
    best_score = -1.0

    try:
        # Fit Randomized Search on the training data with selected features
        random_search.fit(X_train[selected_features], y_train)
        tuning_elapsed_time = time.time() - tuning_start_time
        print(f"Randomized Search completed in {tuning_elapsed_time:.2f} seconds.")

        clf = random_search.best_estimator_
        best_params = random_search.best_params_
        best_score = random_search.best_score_

        print("\nBest parameters found by Randomized Search:")
        print(best_params)
        print(f"Best cross-validation accuracy: {best_score:.4f}")

    except Exception as e:
        print(f"Error during Randomized Search fitting: {e}")
        print("Falling back to training with default XGBoost parameters.")
        clf = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.03, subsample=0.9,
                            colsample_bytree=0.9, use_label_encoder=False, eval_metric='logloss', verbosity=0, random_state=42, n_jobs=-1)
        try:
             clf.fit(X_train[selected_features], y_train)
             best_params = "Default (Tuning Failed)"
        except Exception as fit_e:
             print(f"Error fitting fallback model: {fit_e}")
             print("Cannot train any model. Exiting.")
             return

    if clf is not None:
        print("\nEvaluating final model on the test set...")
        X_test_selected = X_test[selected_features]
        y_pred = clf.predict(X_test_selected)
        evaluate(y_test, y_pred)
    else:
        print("Final model could not be trained or loaded. Skipping evaluation.")

    overall_elapsed_time = time.time() - overall_start_time
    print(f"\nTotal execution time: {overall_elapsed_time:.2f} seconds")
    print("="*50)

if __name__ == "__main__":
    train_file_path = "kdd_train.csv"
    test_file_path = "kdd_test.csv"
    # Adjust top_k, GWO params in impgwo_optimize call, or n_iter_search/param_distributions in main
    main(train_file_path, test_file_path, top_k=30)


Starting feature selection and model training process with initial top_k=30

Loading data from kdd_train.csv and kdd_test.csv
Training set: 125973 samples, 42 features
Testing set:  22544 samples, 42 features

Encoding categorical features...
Encoding target variable...
Finalizing target variable encoding (0-indexed, contiguous)...
Number of final classes in training data: 23
Number of final classes in testing data: 22
Scaling numerical features...
Checking class distribution...
Class distribution before balancing: {0: 67343, 1: 41214, 2: 890, 3: 3599, 4: 2931, 5: 892, 6: 1493, 7: 3633, 8: 2646, 9: 201, 10: 956, 11: 53, 12: 8, 13: 7, 14: 10, 15: 30, 16: 11, 17: 20, 18: 4, 19: 18, 20: 9, 21: 2, 22: 3}
Skipping SMOTE. Unique classes: 23, Min class size: 2
Preprocessing completed in 0.84 seconds

Applying multiple feature ranking methods...
Computing Chi-squared statistics...
Computing Median Absolute Deviation...
Computing Pearson Correlation Coefficients...
Computing Mutual Information